# Kết quả thu thập và chất lượng dữ liệu

Phần này trình bày lần lượt **tiến độ xử lý đầu vào**, **dữ liệu thu được** và **chất lượng đầu ra**. Ngày đối chiếu artifacts: **04/11/2026**. Những run được lưu trước ngày này mô tả trạng thái tại thời điểm xuất kết quả, không mặc định bao gồm các lần chạy/sửa sau đó.

## 1. Tiến độ xử lý 5.000 ID của từng thành viên

**Theo tổng hợp tiến độ do nhóm cung cấp, năm thành viên đã chạy hết dải cá nhân, tổng cộng 25.000 ID được giao.** Thông tin này được ghi ở cột “Đã chạy theo báo cáo nhóm”. Các cột tiếp theo chỉ điền khi có bằng chứng audit đối chiếu được trên đúng dải, để phân biệt tiến độ nhóm báo cáo với mức xác minh của bộ kết quả hiện có.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, HTML
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '_config.yml').exists())
sys.path.insert(0, str(ROOT / 'utils'))
from preliminary_results import (load_members, progress_rows, assignment_outcomes,
    quality_rows, error_groups, performance_rows, run_range, session_members, coverage_rows)
members = load_members(ROOT)
sessions = session_members(ROOT, members)
def table(rows):
    frame = pd.DataFrame(rows, dtype=object)
    frame = frame.where(frame.notna(), 'Chưa đủ bằng chứng')
    def number(value):
        if isinstance(value, float):
            return f'{value:,.2f}'.replace(',', '_').replace('.', ',').replace('_', '.')
        if isinstance(value, int) and not isinstance(value, bool):
            return f'{value:,}'.replace(',', '.')
        return str(value)
    display(HTML(frame.to_html(index=False, escape=True, border=0, classes='dataframe',
                              formatters={column:number for column in frame.columns})))


In [ ]:
table(progress_rows(members))

TV2 và TV5 có record cho đủ 5.000 ID trong audit đã nộp, không còn ID chưa ghi nhận ở hai dải này. TV2 không có lookup chưa giải quyết; TV5 còn bốn lookup lỗi. Đối với TV1/TV3/TV4, bảng chưa điền số xác minh toàn dải vì artifacts hiện có chưa đủ; điều đó **không được diễn giải thành các thành viên chưa chạy**.

“Có record” bao gồm cả lần thử có lỗi. Vì vậy đã chạy qua đủ đầu vào, giải quyết xong mọi lookup và dữ liệu vượt qua validation là những mức hoàn thành khác nhau. Cờ hoàn tất audit có xét validation nên không được dùng thay riêng tiến độ chạy ID.

## 2. Dữ liệu thu được trên phạm vi assignment

Bảng dưới đây sử dụng audit toàn dải đã đối chiếu, không thay số liệu còn thiếu bằng thống kê của run cuối. Số version là số đọc được từ báo cáo kiểm định, có thể chưa đầy đủ nếu kiểm định không đọc được dữ liệu; đây không phải số source tải thành công.

In [ ]:
table(assignment_outcomes(members))

### Kết quả source trong các lượt chạy đã có thống kê

Một corpus 5.000 ID có thể được tạo qua nhiều phiên. Các bảng tiếp theo mô tả **riêng phạm vi của từng lượt resume** có statistics hợp lệ gắn với benchmark, giúp quan sát kết quả xử lý nhưng chưa đại diện toàn assignment. Không cộng chúng với bảng audit toàn dải hoặc với các run chồng lặp.

In [ ]:
valid = [m for m in members if m['authenticated']]
table([{'Thành viên': m['member'], 'Phạm vi lượt đo': run_range(m['benchmark']),
        'Paper phát hiện': m['statistics']['counts']['discovery_success_papers'],
        'Version đã biết': m['statistics']['counts']['expected_versions'],
        'Source tải thành công': m['statistics']['download_statuses'].get('success', 0),
        'Không có source': m['statistics']['download_statuses'].get('no_source', 0),
        'Lỗi API khi tải': m['statistics']['download_statuses'].get('api_error', 0)} for m in valid])
fig, ax = plt.subplots(figsize=(8, 4))
bottom = [0]*len(valid)
for status, label in [('success','Source thành công'), ('no_source','Không có source'), ('api_error','Lỗi API')]:
    values = [m['statistics']['download_statuses'].get(status, 0) for m in valid]
    ax.bar([m['member'] for m in valid], values, bottom=bottom, label=label)
    bottom = [a+b for a,b in zip(bottom,values)]
ax.set_ylabel('Số version trong lượt đo'); ax.set_xlabel('Thành viên — các lượt có phạm vi khác nhau')
ax.legend(); plt.tight_layout(); plt.show()

TV3 có thống kê cho 828 ID ở lượt cuối; TV4 và TV5 cho sáu ID mỗi lượt. Số 0 trong bảng trạng thái này chỉ có ý nghĩa trong chính lượt đo đó. `no_source` là một kết quả được phân loại, còn lỗi API là kết quả cần xử lý hoặc thử lại. TV2 có benchmark nhưng chưa có statistics của snapshot hoàn tất trong ZIP đã nộp.

### Kích thước corpus và references

Kích thước trước/sau lọc dùng cùng tập paper có đủ cặp đo, tổng các version của mỗi paper. Phần loại bỏ gồm hình và các tệp ngoài TeX/BibTeX. References trung bình tính trên các paper truy vấn được, gồm kết quả rỗng hợp lệ. Tất cả số liệu dưới đây vẫn thuộc các lượt đo đã nêu, chưa phải trung bình toàn bộ nhóm.

In [ ]:
table([{'Thành viên': m['member'], 'Paper có cặp đo kích thước': m['statistics']['sizes']['paired_papers'],
        'Trước lọc (bytes/paper)': m['statistics']['sizes']['mean_uncompressed_before_per_paper_bytes'],
        'Sau lọc (bytes/paper)': m['statistics']['sizes']['mean_filtered_per_paper_bytes'],
        'References thu được': m['statistics']['counts']['references'],
        'References/paper truy vấn được': m['statistics']['references']['mean_per_retrieved_paper']} for m in valid])
table([{'Thành viên': m['member'],
        'Truy vấn references thành công (tử/mẫu)': str(m['statistics']['rates']['reference_query_success']['numerator'])+' / '+str(m['statistics']['rates']['reference_query_success']['denominator']),
        'Reference đủ metadata (tử/mẫu)': str(m['statistics']['references']['complete_metadata_rate']['numerator'])+' / '+str(m['statistics']['references']['complete_metadata_rate']['denominator'])} for m in valid])

## 3. Chất lượng dữ liệu sau thu thập

Validator kiểm tra tính nhất quán giữa phiên bản, raw source, corpus, metadata, references và checkpoint. Bảng này chỉ dùng audit toàn dải đã đối chiếu. Tỷ lệ đạt có mẫu số là số paper **đã được kiểm định**, không phải 5.000 ID được giao và không bao gồm các lookup chưa phát hiện paper.

In [ ]:
table(quality_rows(members))

Kết quả hiện có cho thấy TV2 và TV5 đã có dữ liệu được kiểm định trên các paper phát hiện được, nhưng còn paper chưa đạt. Validation fail không đồng nghĩa toàn bộ nội dung paper đều mất hoặc không thể dùng: nguyên nhân có thể ở một version, một stage, đường dẫn/checkpoint hoặc quy tắc kiểm tra. Cần rà soát theo nhóm vấn đề trước khi chốt corpus cho MS2.

Audit TV2 ghi cờ hoàn tất chưa nhất quán với số paper fail; chi tiết kỹ thuật được giữ ở phụ lục. Điểm này ảnh hưởng kết luận về chất lượng/hoàn tất audit, không phủ nhận bằng chứng 5.000 ID đã có record.

### Các nhóm vấn đề ghi nhận

Mỗi paper chỉ đếm một lần trong từng nhóm, nhưng một paper có thể thuộc nhiều nhóm. Vì vậy không cộng các cột để suy tổng paper fail. Đây là phân loại thông báo validator, chưa khẳng định nguyên nhân gốc hoặc trách nhiệm của người chạy.

In [ ]:
groups = error_groups(members)
table(groups)
fig, ax = plt.subplots(figsize=(9, 5))
labels = [r['Thành viên']+' — '+r['Nhóm vấn đề'] for r in groups]
ax.barh(labels, [r['Paper bị ảnh hưởng'] for r in groups])
ax.invert_yaxis(); ax.set_xlabel('Paper bị ảnh hưởng trong từng nhóm (có thể chồng lặp)')
plt.tight_layout(); plt.show()

## 4. Phạm vi kết luận

Nhóm báo cáo đã chạy đủ 5.000 ID/người. Bộ artifacts hiện cho phép đối chiếu đầy đủ record đầu vào của TV2/TV5 và xem kết quả một số phiên của TV3/TV4/TV5. Chưa đủ bằng chứng để công bố số paper/source hợp lệ và tỷ lệ thành công cuối cùng cho toàn bộ 25.000 ID. Bổ sung audit và thống kê toàn dải sẽ hoàn thiện phần xác minh mà không cần mặc định thu thập lại toàn bộ dữ liệu.

Xem [hiệu năng pipeline](performance.ipynb) để phân biệt số đo từng phiên với tổng thời gian thu thập; [phụ lục](audit-appendix.ipynb) lưu nguồn artifacts, run ID và đối chiếu audit.